# B6 (Partie 4) — Explicabilité des détections avec SHAP

**Prérequis** : TP4 exécuté — checkpoints `ae_v3_best.keras` et banque PatchCore disponibles.

Ce TP applique SHAP (SHapley Additive exPlanations) pour répondre à la question production :
> *Pourquoi cette image est-elle classée défectueuse ?*

| § | Analyse | Explainer |
|---|---|---|
| §1 | Attribution **pixel par pixel** — auto-encodeur MSE+SSIM | `GradientExplainer` |
| §2 | Comparaison SHAP vs carte d'erreur de reconstruction | — |
| §3 | Attribution par **superpixels** — PatchCore-lite | `PartitionExplainer` |
| §4 | **Importance globale** des features — classifieur méta | `TreeExplainer` + beeswarm |
| §5 | Limites de l'explicabilité | — |

## §0 — Setup

In [ ]:
import os, sys
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import shap

sys.path.insert(0, str(Path("src")))
from indusense.vision.dataset import load_train_val, load_test, load_masks
from indusense.vision.model   import mse_ssim_loss
from indusense.vision.anomaly import reconstruction_errors, calibrate_threshold

BOTTLE_ROOT = Path("bottle")
SEED        = 42
Path("figures").mkdir(exist_ok=True)

X_train, X_val  = load_train_val(BOTTLE_ROOT, val_ratio=0.2, seed=SEED)
X_test, y_test, test_classes = load_test(BOTTLE_ROOT)

defect_classes = sorted(set(c for c in test_classes if c != "good"))
test_arr       = np.array(test_classes)

print(f"SHAP {shap.__version__}  |  TF {tf.__version__}")
print(f"Train={X_train.shape[0]}  Val={X_val.shape[0]}  Test={X_test.shape[0]}")
print(f"Défauts : {defect_classes}")

In [ ]:
# ── Auto-encodeur v3 (MSE+SSIM, ratio≈12) ────────────────────────────────────
custom_loss = mse_ssim_loss(alpha=0.8)
model_v3 = keras.models.load_model(
    "checkpoints/ae_v3_best.keras",
    custom_objects={custom_loss.__name__: custom_loss},
)
print(f"Auto-encodeur v3 : {model_v3.count_params():,} paramètres")

errors_val_v3  = reconstruction_errors(model_v3, X_val)
errors_test_v3 = reconstruction_errors(model_v3, X_test)
threshold_v3   = calibrate_threshold(errors_val_v3, method="percentile", percentile=99)

# ── PatchCore-lite (reconstruction déterministe, SEED=42) ────────────────────
base = keras.applications.ResNet50(
    include_top=False, input_shape=(256, 256, 3), weights="imagenet"
)
feat_model = keras.Model(base.input, base.get_layer("conv3_block4_out").output)
feat_model.trainable = False
F = 512  # canaux conv3_block4_out de ResNet50

print("Extraction features train (PatchCore)...")
feats_train  = feat_model.predict(X_train, batch_size=8, verbose=1)
patches_train = feats_train.reshape(-1, F)

rng = np.random.default_rng(SEED)
n_coreset   = max(1000, len(patches_train) // 10)
idx_core    = rng.choice(len(patches_train), size=n_coreset, replace=False)
memory_bank = patches_train[idx_core]

nn_pc = NearestNeighbors(n_neighbors=1, algorithm="ball_tree", n_jobs=-1)
nn_pc.fit(memory_bank)
print(f"PatchCore prêt — {memory_bank.shape[0]} patches en mémoire")

## §1 — SHAP Gradient : attribution pixel par pixel (auto-encodeur)

**GradientExplainer** utilise *Expected Gradients* : intégration des gradients de
`∂score/∂pixel` sur un fond aléatoire d'images saines.

- **Score** = MSE(image, reconstruction) → scalaire par image
- **Fond** = 50 images de validation (normales)
- **Résultat** = carte d'attribution de même taille que l'image
  - rouge → ce pixel *augmente* le score d'anomalie
  - bleu → ce pixel *diminue* le score d'anomalie

> ⚠ SHAP mesure l'effet causal des pixels d'**entrée** sur le score,
> contrairement à la carte d'erreur qui montre où le **décodeur** s'est trompé.

In [ ]:
# Scorer : auto-encodeur → somme des erreurs au carré
# reduce_sum (pas reduce_mean) : évite la dilution du gradient sur 196 608 pixels
inp_s   = keras.Input(shape=(256, 256, 3), name="scorer_in")
recon_s = model_v3(inp_s)
diff_s  = keras.layers.Subtract(name="residual")([inp_s, recon_s])
mse_s   = keras.layers.Lambda(
    lambda d: tf.expand_dims(tf.reduce_sum(tf.square(d), axis=[1, 2, 3]), 1),
    name="sse_score",
)(diff_s)
scorer_model = keras.Model(inp_s, mse_s, name="sse_scorer")

# Fond = tout le val set si < 50 images disponibles
n_bg       = min(50, len(X_val))
bg_idx     = np.random.default_rng(SEED).choice(len(X_val), size=n_bg, replace=False)
background = X_val[bg_idx]

# Images défectueuses (1 par classe)
idx_defect = []
for cls in defect_classes:
    candidates = np.where(test_arr == cls)[0]
    if len(candidates):
        idx_defect.append(int(candidates[0]))
idx_defect     = idx_defect[:5]
imgs_defect    = X_test[idx_defect]
classes_defect = test_arr[idx_defect]

print(f"Background : {n_bg} images  |  Images analysées : {list(classes_defect)}")
print("Calcul GradientExplainer (≈ 1-2 min)...")
explainer_ae = shap.GradientExplainer(scorer_model, background)
shap_vals_ae = explainer_ae.shap_values(imgs_defect)

# Normaliser le format selon SHAP 0.52 : array (N,H,W,C,1) ou liste [(N,H,W,C)]
if isinstance(shap_vals_ae, list):
    sv_ae = shap_vals_ae[0]
elif np.array(shap_vals_ae).ndim == 5:
    sv_ae = np.array(shap_vals_ae)[:, :, :, :, 0]
else:
    sv_ae = np.array(shap_vals_ae)

shap_maps_ae = np.abs(sv_ae).sum(axis=-1)   # (N, H, W)
print(f"sv_ae shape      : {sv_ae.shape}")
print(f"SHAP maps shape  : {shap_maps_ae.shape}")
print(f"Valeur max SHAP  : {shap_maps_ae.max():.4f}  (doit être >> 0)")

In [ ]:
# shap.image_plot attend une liste [array (N, H, W, C)]
shap_for_plot = [sv_ae] if not isinstance(shap_vals_ae, list) else shap_vals_ae
shap.image_plot(shap_for_plot, imgs_defect, show=False)
plt.suptitle("SHAP Gradient — attribution pixel (auto-encodeur MSE+SSIM)", y=1.01, fontsize=12)
plt.savefig("figures/tp5_01_shap_gradient.png", bbox_inches="tight", dpi=100)
plt.show()
print("Sauvegardé : figures/tp5_01_shap_gradient.png")

## §2 — Comparaison : carte d'erreur vs carte SHAP

| Visualisation | Question | Espace |
|---|---|---|
| Carte d'erreur | Où le **décodeur** a-t-il mal reconstruit ? | Espace **OUTPUT** |
| SHAP map | Quels pixels d'**entrée** causent le score élevé ? | Espace **INPUT** |

Les deux cartes peuvent diverger : un pixel fortement défectueux peut être
partiellement masqué par le décodeur mais avoir une forte attribution SHAP.

In [ ]:
fig, axes = plt.subplots(len(idx_defect), 3, figsize=(12, 4.5 * len(idx_defect)))

col_titles = [
    "Image originale",
    "Erreur reconstruction\n(décodeur → output)",
    "SHAP map (|valeurs| / canal RGB)\n(gradient → input)",
]
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontweight="bold", fontsize=9)

recons = model_v3.predict(imgs_defect, batch_size=5, verbose=0)

for row, (img, recon, cls, shap_map) in enumerate(
    zip(imgs_defect, recons, classes_defect, shap_maps_ae)
):
    error_map = np.mean((img - recon) ** 2, axis=-1)  # (256, 256)

    axes[row, 0].imshow(img)
    axes[row, 0].set_ylabel(cls, rotation=0, labelpad=65, va="center", fontsize=9)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(error_map, cmap="hot")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(shap_map, cmap="Reds")  # shap_map : (256, 256)
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig("figures/tp5_02_shap_vs_error.png", bbox_inches="tight", dpi=100)
plt.show()
print("Sauvegardé : figures/tp5_02_shap_vs_error.png")

## §3 — SHAP Partition : superpixels sur PatchCore

PatchCore n'est pas différentiable (kNN, pas de gradient) → on utilise
**PartitionExplainer** (appelé automatiquement via `shap.Explainer` avec un masker image).

**Principe :**
1. Partitionner l'image en superpixels (régions cohérentes)
2. Masquer/inpainter des groupes de superpixels
3. Mesurer l'effet sur le score PatchCore → valeur de Shapley par superpixel

> ⏱ ~2-4 min par image (500 évaluations du modèle)

In [ ]:
def patchcore_scorer(imgs: np.ndarray) -> np.ndarray:
    """Scorer batch PatchCore. imgs: (n, H, W, C) float32 [0,1]."""
    feats = feat_model(tf.cast(imgs, tf.float32), training=False).numpy()
    scores = []
    for f in feats:
        p = f.reshape(-1, F)
        dists, _ = nn_pc.kneighbors(p)
        scores.append(float(dists.max()))
    return np.array(scores, dtype=np.float32)

# Masker : blur(128,128) ne nécessite pas OpenCV (inpaint_telea en dépend)
masker_pc    = shap.maskers.Image("blur(128,128)", shape=X_test[0].shape)
explainer_pc = shap.Explainer(patchcore_scorer, masker_pc)

# 3 images défectueuses (lent — ≈ 2-4 min par image)
imgs_pc    = imgs_defect[:3]
classes_pc = classes_defect[:3]

print(f"Analyse PatchCore SHAP pour : {list(classes_pc)}")
shap_vals_pc = explainer_pc(imgs_pc, max_evals=500, batch_size=50)
print(f"SHAP values shape : {shap_vals_pc.values.shape}")
print(f"Valeur max SHAP   : {np.abs(shap_vals_pc.values).max():.6f}  (doit être > 0)")

In [ ]:
fig, axes = plt.subplots(len(imgs_pc), 2, figsize=(9, 4 * len(imgs_pc)))

axes[0, 0].set_title("Image originale", fontweight="bold")
axes[0, 1].set_title("SHAP superpixels\n(rouge = ↑ score anomalie)", fontweight="bold")

for row, (img, cls) in enumerate(zip(imgs_pc, classes_pc)):
    # Agréger sur les canaux : (H, W)
    sv = shap_vals_pc.values[row].sum(axis=-1)  # (H, W)
    v_abs = np.abs(sv).max() or 1.0

    axes[row, 0].imshow(img)
    axes[row, 0].set_ylabel(cls, rotation=0, labelpad=65, va="center", fontsize=9)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(sv, cmap="RdBu_r", vmin=-v_abs, vmax=v_abs)
    axes[row, 1].axis("off")

plt.tight_layout()
plt.savefig("figures/tp5_03_shap_patchcore.png", bbox_inches="tight", dpi=100)
plt.show()
print("Sauvegardé : figures/tp5_03_shap_patchcore.png")

## §4 — Importance globale des features (TreeExplainer + beeswarm)

On extrait un vecteur de **8 features de haut niveau** par image, on entraîne
un `RandomForestClassifier`, puis on applique `shap.TreeExplainer` pour obtenir
les valeurs de Shapley exactes (algorithme polynomial sur les arbres).

**Features :**

| Feature | Description |
|---|---|
| `mse_score` | Erreur MSE moyenne (auto-encodeur v3) |
| `ssim_score` | SSIM image vs reconstruction |
| `patchcore_score` | Distance max kNN (PatchCore) |
| `max_error` | Max pixel error (carte d'erreur) |
| `p95_error` | Percentile 95 de la carte d'erreur |
| `std_error` | Écart-type de la carte d'erreur |
| `mean_intensity` | Luminosité moyenne de l'image |
| `std_intensity` | Contraste de l'image |

Le beeswarm SHAP répond : *quelle feature discrimine le plus les images saines des images défectueuses ?*

In [ ]:
print("Extraction des features...")

# Reconstructions en batch
recons_test  = model_v3.predict(X_test,  batch_size=16, verbose=1)
error_maps   = np.mean((X_test - recons_test) ** 2, axis=-1)  # (N, 256, 256)

# SSIM batch
ssim_scores = tf.image.ssim(
    tf.cast(X_test,        tf.float32),
    tf.cast(recons_test,   tf.float32),
    max_val=1.0,
).numpy()  # (N,)

# Scores PatchCore test
print("Scores PatchCore test...")
feats_test    = feat_model.predict(X_test, batch_size=8, verbose=1)
scores_pc_test = []
for f in feats_test:
    p = f.reshape(-1, F)
    dists, _ = nn_pc.kneighbors(p)
    scores_pc_test.append(float(dists.max()))
scores_pc_test = np.array(scores_pc_test, dtype=np.float32)

FEATURE_NAMES = [
    "mse_score", "ssim_score", "patchcore_score",
    "max_error", "p95_error", "std_error",
    "mean_intensity", "std_intensity",
]

X_feat = np.column_stack([
    errors_test_v3,
    ssim_scores,
    scores_pc_test,
    error_maps.max(axis=(1, 2)),
    np.percentile(error_maps, 95, axis=(1, 2)),
    error_maps.std(axis=(1, 2)),
    X_test.mean(axis=(1, 2, 3)),
    X_test.std(axis=(1, 2, 3)),
]).astype(np.float32)

print(f"Features shape : {X_feat.shape}  — {FEATURE_NAMES}")

In [ ]:
clf = RandomForestClassifier(n_estimators=300, max_depth=None, random_state=SEED, n_jobs=-1)

cv_auroc = cross_val_score(clf, X_feat, y_test, cv=5, scoring="roc_auc")
print(f"RandomForest CV AUROC : {cv_auroc.mean():.3f} ± {cv_auroc.std():.3f}")

clf.fit(X_feat, y_test)

# TreeExplainer — exact polynomial sur les arbres
explainer_tree = shap.TreeExplainer(clf)
shap_vals_tree = explainer_tree.shap_values(X_feat)

# SHAP 0.46+ peut retourner un array 3D (N, n_feat, n_classes)
# ou une liste [classe0, classe1] — on normalise
if isinstance(shap_vals_tree, list):
    sv_anomaly = shap_vals_tree[1]        # liste → prendre classe 1
else:
    sv_anomaly = shap_vals_tree[:, :, 1]  # array 3D → slice classe 1

print(f"sv_anomaly shape : {sv_anomaly.shape}  (attendu : {X_feat.shape})")

# Beeswarm
plt.figure(figsize=(9, 5))
shap.summary_plot(
    sv_anomaly, X_feat,
    feature_names=FEATURE_NAMES,
    plot_type="dot",
    show=False,
)
plt.title("Importance globale — SHAP beeswarm (RandomForest, classe = défaut)")
plt.tight_layout()
plt.savefig("figures/tp5_04_beeswarm.png", bbox_inches="tight", dpi=100)
plt.show()
print("Sauvegardé : figures/tp5_04_beeswarm.png")

# Waterfall d'un exemple défectueux
first_defect_idx = int(np.where(y_test == 1)[0][0])
base_val = (
    explainer_tree.expected_value[1]
    if isinstance(explainer_tree.expected_value, (list, np.ndarray))
    else explainer_tree.expected_value
)
explanation = shap.Explanation(
    values=sv_anomaly[first_defect_idx],
    base_values=base_val,
    data=X_feat[first_defect_idx],
    feature_names=FEATURE_NAMES,
)
shap.waterfall_plot(explanation, show=False)
plt.title(f"Waterfall — image défectueuse ({test_arr[first_defect_idx]})")
plt.tight_layout()
plt.savefig("figures/tp5_04b_waterfall.png", bbox_inches="tight", dpi=100)
plt.show()
print("Sauvegardé : figures/tp5_04b_waterfall.png")

## §5 — Analyse et limites de l'explicabilité

### Ce que SHAP apporte

- **GradientExplainer** : attribution causale des pixels d'entrée → utile pour valider
  que le modèle regarde la bonne zone (défaut, pas fond/bouchon).
- **PartitionExplainer** : applicable à tout scorer noir, y compris PatchCore.
  Les superpixels sont plus faciles à interpréter humainement que les pixels.
- **TreeExplainer** : importance globale → permet de répondre *quelle famille de features*
  (reconstruction, structure SSIM, feature-based) est la plus discriminante.

### Limites

| Limite | Explication |
|---|---|
| **Coût calcul** | GradientExplainer ≈ O(n_background × n_images) ; PartitionExplainer ≈ 500 forward-passes/image |
| **Corrélation ≠ causalité** | SHAP mesure des contributions marginales, pas des relations causales |
| **Dépendance au fond** | GradientExplainer est sensible au choix des images de fond |
| **Modèle à expliquer** | Si le modèle est mauvais (AUROC faible), les SHAP values expliquent un mauvais modèle |
| **Taille des images** | 256×256 = 196 608 features → PartitionExplainer contraint à des groupes (superpixels) |

### Conclusion

L'explicabilité est complémentaire à la performance, non substituable.
En contexte industriel, combiner :
- une **bonne performance** (PatchCore, AUROC ≥ 0.85)
- une **carte de localisation** (superpixels SHAP ou heatmap de reconstruction)
- un **rapport de confiance** (score brut + seuil)

...permet à un opérateur humain de valider ou rejeter la décision du système.